In [ ]:
# Lecture 7 — What Is a Partition
# Spark 101 · Module 3 · Lecture 7
#
# Import this file into a Databricks workspace as a notebook.
# Databricks recognizes `# COMMAND ----------` as a cell boundary on import.

In [ ]:
# Load the sample dataset — pre-loaded in every Databricks workspace.
df = spark.read.table("samples.nyctaxi.trips")

In [ ]:
# How many partitions does Spark see?
# Same information will show up as "task count" in the Spark UI.
print(f"Partitions: {df.rdd.getNumPartitions()}")

In [ ]:
# Trigger a job — then flip to the Spark UI.
# The number of tasks in the stage = the number of partitions above.
print(f"Row count: {df.count():,}")

In [ ]:
# A tiny DataFrame from a Python list — usually a single partition.
tiny = spark.createDataFrame([(i,) for i in range(10)], ["n"])
print(f"Tiny DataFrame partitions: {tiny.rdd.getNumPartitions()}")

In [ ]:
# Increase parallelism: repartition into 8 even partitions (a shuffle).
more = df.repartition(8)
print("after repartition(8):", more.rdd.getNumPartitions())

In [ ]:
# Reduce without a full shuffle: coalesce down to 2.
print("after coalesce(2):", more.coalesce(2).rdd.getNumPartitions())

In [ ]:
# How many rows landed in each partition? (watch for skew)
from pyspark.sql.functions import spark_partition_id, count
(more.withColumn("pid", spark_partition_id())
     .groupBy("pid").agg(count("*").alias("rows"))
     .orderBy("pid").show())

In [ ]:
# The default parallelism the cluster starts from.
print("default parallelism:", spark.sparkContext.defaultParallelism)